# Phase 6 — Senior Data Scientist Personality Modeling & Interpretability Walkthrough
**Project**: InsightPath / RUSTY WOLVES  
**Evaluation Pillar**: Data Mining / Machine Learning Skills (30 Marks), Model Interpretability, Ethical Safeguards  
**Authoritative Inputs**:  (=161$) &  (=152$)


In [1]:
# 1. Environment & Library Setup
import sys
sys.path.insert(0, "../..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.modeling.sds.data import load_sds_primary_data, load_sds_sensitivity_data, audit_sds_duplicates
from src.modeling.sds.pipelines import get_sds_models
from src.modeling.sds.cross_validation import run_sds_grouped_cross_validation
from src.modeling.sds.evaluation import build_model_performance_summary
from src.modeling.sds.interpretability import extract_logistic_interpretability, extract_decision_tree_rules, compute_permutation_importance_on_held_out
from src.modeling.sds.sensitivity import run_sds_sensitivity_comparison
from src.modeling.sds.error_analysis import run_sds_error_analysis
print("Libraries loaded successfully!")


In [2]:
# 2. Data Ingestion & Schema Validation
X_prim, y_prim, groups_prim = load_sds_primary_data("../../data/processed/sds_processed.csv")
print("Primary Cohort Shape:", X_prim.shape)
print("Class Distribution:", y_prim.value_counts().to_dict())
print("Unique Subject IDs:", groups_prim.nunique())


In [3]:
# 3. Duplicate Audit (18 rows, 9 duplicate IDs)
df_dups = audit_sds_duplicates("../../data/processed/sds_processed.csv")
print(f"Identified {df_dups['id'].nunique()} duplicated subject IDs across {len(df_dups)} total rows.")
df_dups[["id", "neuroticism", "extraversion", "openness_to_experience", "agreeableness", "conscientiousness", "success_classification_high_low"]].head(6)


In [4]:
# 4. Group-Aware Cross-Validation Execution (StratifiedGroupKFold on ID, 25 splits)
fold_df, oof_df, _ = run_sds_grouped_cross_validation(X_prim, y_prim, groups_prim, n_splits=5, seeds=[42, 43, 44, 45, 46])
print(f"Evaluated {len(fold_df)} total split evaluations across 7 candidate models.")


In [5]:
# 5. Model Performance Scorecard
perf_df = build_model_performance_summary(fold_df)
perf_df[["model_name", "roc_auc_mean", "roc_auc_std", "macro_f1_mean", "accuracy_mean", "brier_score_mean"]]


In [6]:
# 6. Standardized Logistic Odds Ratios (Champion Model: Logistic L2)
coef_df, or_df, stab_df = extract_logistic_interpretability(X_prim, y_prim, groups_prim, model_key="Logistic_Regression_L2")
or_df[["trait_dimension", "standardized_coef_beta", "adjusted_odds_ratio", "or_ci_lower_95", "or_ci_upper_95", "relative_importance_rank"]]


In [7]:
# 7. Decision Tree Programmatic Rules (CART, max_depth=3)
rules_df, comp_df = extract_decision_tree_rules(X_prim, y_prim)
print("Tree Complexity:", comp_df.to_dict(orient="records"))
rules_df[["leaf_id", "rule_path", "n_samples", "prob_high_success", "predicted_label"]]


In [8]:
# 8. Permutation Feature Importance on Held-Out Validation Folds
rf_imp_df, feat_stab_df = compute_permutation_importance_on_held_out(X_prim, y_prim, groups_prim, model_key="Random_Forest")
rf_imp_df[["trait_dimension", "mean_permutation_importance", "std_permutation_importance", "rank"]]


In [9]:
# 9. Out-of-Fold Error Analysis
err_summary, err_profile = run_sds_error_analysis(oof_df, X_prim, groups_prim, model_name="Logistic_Regression_L2")
err_summary


In [10]:
# 10. Sensitivity Robustness Analysis (Primary N=161 vs Deduplicated N=152)
X_sens, y_sens, _ = load_sds_sensitivity_data("../../data/processed/sds_sensitivity_deduplicated.csv")
sens_perf, sens_coef, sens_imp, verdict = run_sds_sensitivity_comparison(perf_df, X_sens, y_sens)
print("Formal Robustness Verdict:", verdict)
sens_perf[["model_name", "roc_auc_mean_primary", "roc_auc_mean_sensitivity", "delta_roc_auc", "stability_status"]]


In [11]:
# 11. Ethical Interpretation & Governance Guardrails
print("CRITICAL ETHICAL DIRECTIVE:")
print("- Big Five personality models reflect empirical correlations with observed consulting ratings.")
print("- Strict prohibition against personality-based hiring gates, termination scoring, or exclusion.")
print("- Personality findings serve exclusively as coaching, self-awareness, and mentoring guidance.")
